In [15]:
# =============================================================================
# Polished Box Plot: 1 Row x 6 Columns | Python Equivalent
# Output: Editable Vector PDF and SVG for Adobe Illustrator
# =============================================================================

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import string

# ── 1. Settings & Illustrator Vector Text Fixes ──────────────────────────────
# Ensure text is exported as editable text objects, NOT paths
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
plt.rcParams['svg.fonttype'] = 'none'

BASE_FONT = "Arial"
plt.rcParams['font.family'] = BASE_FONT

# Scale constraints (Base: 190x45 mm -> Scaled 3x)
SCALE = 3
WIDTH_IN = (190 * SCALE) / 25.4   # mm to inches
HEIGHT_IN = (45 * SCALE) / 25.4   # mm to inches

# Scaled Font Sizes
FONT_TITLE = 6 * SCALE
FONT_AXIS_LABEL = 6 * SCALE
FONT_TICK = 5 * SCALE
FONT_TAG = 7 * SCALE

# ── 2. Data Loading & Reshaping (Wide to Long) ───────────────────────────────
def load_data(path, entity_type):
    # Use pd.read_excel(path) if your local files are .xlsx instead of .csv
    df = pd.read_excel(path) 
    
    # Reshape the data: unpivot the metric columns into rows
    df_melted = df.melt(
        id_vars=['query', 'method'], 
        value_vars=['precision', 'recall', 'f1', 'accuracy', 'specificity', 'latency'],
        var_name='metric', 
        value_name='value'
    )
    
    # Rename 'query' to 'entity' to match our plotting logic
    df_melted = df_melted.rename(columns={'query': 'entity'})
    df_melted['entity_type'] = entity_type
    
    # Ensure numeric values
    df_melted['value'] = pd.to_numeric(df_melted['value'], errors='coerce')
    
    return df_melted

# Update these paths to match where your files are saved locally
df_all = pd.concat([
    load_data("disease_metric_complete.xlsx", "Disease"),
    load_data("drug_metric_complete.xlsx",    "Drug"),
    load_data("gene_metric_complete.xlsx",    "Gene")
], ignore_index=True)

# ── 3. Label Mappings & Colors ───────────────────────────────────────────────
updated_names = {
    "BC-FuzzyEq": "BC-FuzzyEq",
    "BC-Curated": "BC-Curated",
    "BC-EmbedEq": "BC-EmbedEq",
    "BC-Final":   "BC-Final",
    "gpt":        "GPT-5-Nano",
    "grok":       "Grok-4.1 (non-reasoning)",
    "gemini":     "Gemini-2.5-Flash-Lite",
    "llama":      "LLaMA-3.3-70B-Versatile"
}

# Color logic: Teal if 'BC' is in the name, otherwise Grey
palette_fill = {v: "#80cbc4" if "BC" in v else "#e6e6e6" for v in updated_names.values()}

# ── 4. Figure Setup ──────────────────────────────────────────────────────────
fig, axes = plt.subplots(nrows=1, ncols=6, figsize=(WIDTH_IN, HEIGHT_IN))
# Adjust spacing to fit the 45-degree angled text
fig.subplots_adjust(bottom=0.35, top=0.85, left=0.03, right=0.98, wspace=0.3)

panels = [
    ("recall", "Disease", "Recall",   "Disease"),
    ("recall", "Drug",    "Recall",   "Drug"),
    ("recall", "Gene",    "Recall",   "Gene"),
    ("f1",     "Disease", "F1 Score", "Disease"),
    ("f1",     "Drug",    "F1 Score", "Drug"),
    ("f1",     "Gene",    "F1 Score", "Gene")
]

# ── 5. Plot Generation Loop ──────────────────────────────────────────────────
for i, (ax, (metric, entity, y_title, p_title)) in enumerate(zip(axes, panels)):
    
    # Filter and map names
    df_plot = df_all[(df_all['metric'] == metric) & (df_all['entity_type'] == entity)].copy()
    df_plot['method_clean'] = df_plot['method'].map(updated_names)
    df_plot = df_plot.dropna(subset=['method_clean'])
    
    # Sort by median descending
    medians = df_plot.groupby('method_clean')['value'].median().sort_values(ascending=False)
    order = medians.index.tolist()
    
    # 1. Background Anchor Line
    ax.axhline(1.0, color='#cccccc', linestyle='--', linewidth=1.0 * SCALE, zorder=0)

    # 2. Box Plot
    sns.boxplot(
        data=df_plot, 
        x='method_clean', 
        y='value', 
        order=order,
        palette=palette_fill,
        hue='method_clean', # hue required for modern seaborn palettes
        legend=False,
        width=0.6,
        linewidth=0.5 * SCALE,
        flierprops=dict(marker='o', markersize=2*SCALE, color='#999999', alpha=0.5, markeredgecolor='none'),
        medianprops=dict(color="#4d4d4d", linewidth=1.5*SCALE),
        boxprops=dict(alpha=0.85, edgecolor="#4d4d4d"),
        whiskerprops=dict(color="#4d4d4d"),
        capprops=dict(color="#4d4d4d"),
        ax=ax,
        zorder=3
    )

    # 3. Y-Axis Formatting
    ax.set_ylim(-0.02, 1.05)
    ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0])
    ax.set_yticklabels(["0", "0.25", "0.5", "0.75", "1"], fontsize=FONT_TICK)
    ax.set_ylabel(y_title, fontsize=FONT_AXIS_LABEL, fontweight='bold', labelpad=2*SCALE)

    # 4. X-Axis Formatting
    ax.set_xlabel("")
    ax.set_xticks(range(len(order)))
    # Rotate 45 deg, align right, set anchor to top for clean tick alignment
    ax.set_xticklabels(order, rotation=45, ha='right', rotation_mode='anchor', fontsize=FONT_TICK)

    # 5. Titles & Tags
    ax.set_title(p_title, fontsize=FONT_TITLE, fontweight='bold', pad=4*SCALE)
    
    # Add A, B, C, D, E, F tag at top left of each panel
    ax.text(-0.25, 1.15, string.ascii_uppercase[i], transform=ax.transAxes, 
            fontsize=FONT_TAG, fontweight='bold', va='top', ha='right')

    # 6. Clean Spines (Theme Minimal)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_linewidth(0.5 * SCALE)
    ax.spines['left'].set_linewidth(0.5 * SCALE)
    ax.tick_params(width=0.5 * SCALE, length=2 * SCALE)

# ── 6. Export ────────────────────────────────────────────────────────────────
pdf_path = "boxplot_1row_python.pdf"
svg_path = "boxplot_1row_python.svg"

plt.savefig(pdf_path, format='pdf', transparent=True, dpi=300)
plt.savefig(svg_path, format='svg', transparent=True)

print(f"Saved {pdf_path} and {svg_path} at 3x scale. Ready for Illustrator.")

findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font f

Saved boxplot_1row_python.pdf and boxplot_1row_python.svg at 3x scale. Ready for Illustrator.


In [19]:
# =============================================================================
# Precision vs Recall — Disease | Drug | Gene
# Style: F1 iso-curves (Sharp Black), L-shaped axes, NO GRID, AI-compatible
# A4 width: 180 mm, height: 50 mm | Arial, size 7 | Editable Vectors
# Summary statistic: median
# =============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.lines as mlines

# ── 0. Font & Illustrator Compatibility Setup ────────────────────────────────
# Forces Matplotlib to export actual text objects, NOT uneditable shapes/paths
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
plt.rcParams['svg.fonttype'] = 'none'

BASE_FONT = "Arial"
plt.rcParams['font.family'] = BASE_FONT
plt.rcParams['font.size'] = 7

# ── 1. Method Aesthetics ─────────────────────────────────────────────────────
method_levels = [
    "BC-Final", "BC-EmbedEq", "BC-FuzzyEq", "BC-Curated",
    "gpt", "grok", "gemini", "llama"
]

method_labels = {
    "BC-Final": "BC-Final",     "BC-EmbedEq": "BC-EmbedEq",
    "BC-FuzzyEq": "BC-FuzzyEq", "BC-Curated": "BC-Curated",
    "gpt": "GPT-4o-mini",       "grok": "Grok-4.1",
    "gemini": "Gemini-2.5",     "llama": "LLaMA-3.3"
}

method_colors = {
    "BC-Final": "#5BBCBF", "BC-EmbedEq": "#5B8C3E",
    "BC-FuzzyEq": "#8FBC5A", "BC-Curated": "#C8D96F",
    "gpt": "#D97B3A", "grok": "#C4572A",
    "gemini": "#B5A020", "llama": "#9B6BBF"
}

# Matplotlib Markers: 's' = square, 'D' = diamond, 'o' = circle
method_shapes = {
    "BC-Final": "s", "BC-EmbedEq": "D",
    "BC-FuzzyEq": "s", "BC-Curated": "o",
    "gpt": "o", "grok": "D",
    "gemini": "o", "llama": "s"
}

# ── 2. Data Loading & Median Summary ─────────────────────────────────────────
def prepare_data(path):
    # Your attached files are CSVs, so we use read_csv
    df = pd.read_excel(path)
    
    # Calculate median of precision and recall for each method
    summ = df.groupby('method')[['precision', 'recall']].median().reset_index()
    summ = summ[summ['method'].isin(method_levels)]
    
    # Rename columns to match plotting logic
    summ = summ.rename(columns={'precision': 'med_precision', 'recall': 'med_recall'})
    return summ

# Replace with your local file paths
s_dis = prepare_data("disease_metric_complete.xlsx")
s_drg = prepare_data("drug_metric_complete.xlsx")
s_gen = prepare_data("gene_metric_complete.xlsx")

# ── 3. Figure Layout Setup ───────────────────────────────────────────────────
WIDTH_IN = 120 / 25.4
HEIGHT_IN = 50 / 25.4

fig, axes = plt.subplots(1, 3, figsize=(WIDTH_IN, HEIGHT_IN))
# Adjusted margins: Tight on top/sides, space at bottom for the shared legend
fig.subplots_adjust(bottom=0.28, top=0.90, left=0.06, right=0.98, wspace=0.15)

panels = [
    (axes[0], s_dis, "Disease"),
    (axes[1], s_drg, "Drug"),
    (axes[2], s_gen, "Gene")
]

# ── 4. Drawing the Panels ────────────────────────────────────────────────────
f1_vals = [0.25, 0.50, 0.75]

for ax, df, title in panels:
    ax.set_facecolor("#FCFCFC") # Very light grey background
    
    # --- A. Draw F1 Iso-curves (Sharp Black) ---
    for f1 in f1_vals:
        r_min = f1 / (2 - f1) + 0.001
        r = np.linspace(r_min, 1.0, 400)
        p = (f1 * r) / (2 * r - f1)
        
        # Only plot where precision <= 1.0
        valid = p <= 1.0
        ax.plot(r[valid], p[valid], color="black", linestyle="--", linewidth=0.5, zorder=1)
        
        # Add F1 Labels cleanly on the curve
        r_max = 0.96
        p_at_r = (f1 * r_max) / (2 * r_max - f1)
        p_label = min(p_at_r, 0.98)
        ax.text(r_max, p_label, str(f1), fontsize=6, color="black", 
                ha='left', va='bottom', zorder=2)

    # --- B. Plot the Points ---
    for method in method_levels:
        row = df[df['method'] == method]
        if row.empty: continue
            
        r_val = row['med_recall'].values[0]
        p_val = row['med_precision'].values[0]
        
        is_bc = (method == "BC-Final")
        size = 35 if is_bc else 20
        z = 4 if is_bc else 3
        
        ax.scatter(r_val, p_val, color=method_colors[method], marker=method_shapes[method],
                   s=size, alpha=0.95, zorder=z)
        
        # --- C. Add Annotation Arrow for BC-Final ---
        if is_bc:
            ann_label = f"BC-Final\n(P={p_val:.2f}, R={r_val:.2f})"
            label_r = max(0.15, r_val - 0.25)
            label_p = max(0.15, p_val - 0.18)
            
            ax.annotate(
                ann_label, 
                xy=(r_val - 0.02, p_val - 0.02), # Point near the marker
                xytext=(label_r, label_p),       # Start arrow from here
                arrowprops=dict(
                    arrowstyle="-|>", 
                    connectionstyle="arc3,rad=-0.2",
                    color=method_colors[method], 
                    lw=0.8, shrinkA=0, shrinkB=2
                ),
                fontsize=6, fontweight='bold', color=method_colors[method],
                ha='center', va='center', linespacing=1.2, zorder=5
            )

    # --- D. Formatting (L-Shaped Axes, No Grid) ---
    ax.set_xlim(0, 1.05)
    ax.set_ylim(0, 1.10)
    ax.set_xticks([0, 0.25, 0.50, 0.75, 1.00])
    ax.set_yticks([0, 0.25, 0.50, 0.75, 1.00])
    ax.set_xticklabels(["0.00", "0.25", "0.50", "0.75", "1.00"], fontsize=6, color="grey")
    
    ax.set_title(title, fontsize=8, fontweight='bold', pad=4)
    ax.set_xlabel("Recall", fontsize=7, fontweight='bold', labelpad=2)
    
    # Hide Y-axis labels on inner plots to keep it clean
    if ax == axes[0]:
        ax.set_ylabel("Precision", fontsize=7, fontweight='bold', labelpad=2)
        ax.set_yticklabels(["0.00", "0.25", "0.50", "0.75", "1.00"], fontsize=6, color="grey")
    else:
        ax.set_yticklabels([])

    # Explicitly remove Top/Right Spines to create the L-shape
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_linewidth(0.5)
    ax.spines['left'].set_linewidth(0.5)
    ax.tick_params(width=0.4, length=2, color="black")

# ── 5. Shared Legend Creation ────────────────────────────────────────────────
# Build proxy artists for the custom legend
legend_elements = [
    mlines.Line2D([], [], color='w', marker=method_shapes[m], 
                  markerfacecolor=method_colors[m], markersize=5.5, 
                  label=method_labels[m]) 
    for m in method_levels
]

# Place legend cleanly centered at the bottom
fig.legend(handles=legend_elements, loc='lower center', ncol=8, 
           frameon=False, fontsize=6, handletextpad=0.2, columnspacing=1.0, 
           bbox_to_anchor=(0.5, 0.0))

# ── 6. Export to PDF & SVG (Editable text logic included) ────────────────────
pdf_path = "precision_recall_nogrid_python.pdf"
svg_path = "precision_recall_nogrid_python.svg"
png_path = "precision_recall_nogrid_python.png"

plt.savefig(pdf_path, format='pdf', transparent=True, dpi=300)
plt.savefig(svg_path, format='svg', transparent=True)
plt.savefig(png_path, format='png', transparent=False, dpi=300, facecolor='white')

print(f"Saved: {pdf_path}, {svg_path}, and {png_path}")

findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font f

Saved: precision_recall_nogrid_python.pdf, precision_recall_nogrid_python.svg, and precision_recall_nogrid_python.png


In [21]:
# =============================================================================
# Latency Distribution Box Plot — Disease | Drug | Gene
# Style: L-shaped axes, NO GRID, 8-color palette, AI-compatible text
# Dimensions: 180 mm x 50 mm (Scaled 3x for high-res downscaling in Illustrator)
# Output: Editable Vector PDF and SVG
# =============================================================================

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ── 1. Settings & Illustrator Vector Text Fixes ──────────────────────────────
# Ensure text is exported as editable text objects, NOT paths
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
plt.rcParams['svg.fonttype'] = 'none'

BASE_FONT = "Arial"
plt.rcParams['font.family'] = BASE_FONT

# Scale constraints (Base: 180x50 mm -> Scaled 3x for crisp editing)
SCALE = 3
WIDTH_IN = (180 * SCALE) / 25.4   # mm to inches
HEIGHT_IN = (50 * SCALE) / 25.4   # mm to inches

# Scaled Font Sizes
FONT_TITLE = 8 * SCALE
FONT_AXIS_TITLE = 7 * SCALE
FONT_TICK = 6 * SCALE

# ── 2. Aesthetics (Matching the PR Plot exactly) ─────────────────────────────
method_levels = [
    "BC-Final", "BC-EmbedEq", "BC-FuzzyEq", "BC-Curated",
    "gpt", "grok", "gemini", "llama"
]

method_labels = {
    "BC-Final": "BC-Final",     "BC-EmbedEq": "BC-EmbedEq",
    "BC-FuzzyEq": "BC-FuzzyEq", "BC-Curated": "BC-Curated",
    "gpt": "GPT-4o-mini",       "grok": "Grok-4.1",
    "gemini": "Gemini-2.5",     "llama": "LLaMA-3.3"
}

method_colors = {
    "BC-Final": "#5BBCBF", "BC-EmbedEq": "#5B8C3E",
    "BC-FuzzyEq": "#8FBC5A", "BC-Curated": "#C8D96F",
    "gpt": "#D97B3A",      "grok": "#C4572A",
    "gemini": "#B5A020",   "llama": "#9B6BBF"
}

# ── 3. Data Loading ──────────────────────────────────────────────────────────
def load_latency_data(path, entity_type):
    df = pd.read_excel(path)
    # Filter only the columns we need, drop any missing latency rows
    df_clean = df[['method', 'latency']].dropna()
    df_clean['entity_type'] = entity_type
    # Map to clean labels
    df_clean['method_clean'] = df_clean['method'].map(method_labels)
    # Ensure correct plotting order
    df_clean['method_clean'] = pd.Categorical(
        df_clean['method_clean'], 
        categories=[method_labels[m] for m in method_levels], 
        ordered=True
    )
    return df_clean

# Replace with your actual paths
df_dis = load_latency_data("disease_metric_complete.xlsx", "Disease")
df_drg = load_latency_data("drug_metric_complete.xlsx", "Drug")
df_gen = load_latency_data("gene_metric_complete.xlsx", "Gene")

# ── 4. Figure Setup ──────────────────────────────────────────────────────────
fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(WIDTH_IN, HEIGHT_IN))

# Generous bottom margin to accommodate the 45-degree method labels
fig.subplots_adjust(bottom=0.35, top=0.88, left=0.06, right=0.98, wspace=0.25)

panels = [
    (axes[0], df_dis, "Disease"),
    (axes[1], df_drg, "Drug"),
    (axes[2], df_gen, "Gene")
]

# ── 5. Plot Generation Loop ──────────────────────────────────────────────────
for ax, df_plot, title in panels:
    ax.set_facecolor("#FCFCFC") # Very light grey background matching PR plot
    
    # 1. Box Plot
    sns.boxplot(
        data=df_plot, 
        x='method_clean', 
        y='latency', 
        palette=list(method_colors.values()), 
        hue='method_clean',
        legend=False,
        width=0.6,
        linewidth=0.5 * SCALE,
        # Semi-transparent outliers so dense clusters don't look like a solid black block
        flierprops=dict(marker='o', markersize=2*SCALE, color='#666666', alpha=0.3, markeredgecolor='none'),
        medianprops=dict(color="black", linewidth=1.5*SCALE),
        boxprops=dict(alpha=0.9, edgecolor="black"),
        whiskerprops=dict(color="black"),
        capprops=dict(color="black"),
        ax=ax,
        zorder=3
    )

    # 2. X-Axis Formatting
    ax.set_xlabel("")
    # Rotate 45 deg, align right, set anchor to top for clean tick alignment
    ax.set_xticklabels(
        [method_labels[m] for m in method_levels], 
        rotation=45, ha='right', rotation_mode='anchor', fontsize=FONT_TICK, color="black"
    )

    # 3. Y-Axis Formatting
    # We let matplotlib auto-scale the Y axis since latency ranges can vary wildly.
    # If you want to force them all to the same scale, you can uncomment the next line:
    # ax.set_ylim(0, 20) 
    ax.set_ylabel("Latency (s)", fontsize=FONT_AXIS_TITLE, fontweight='bold', labelpad=2*SCALE)
    ax.tick_params(axis='y', labelsize=FONT_TICK, colors="black")

    # 4. Titles
    ax.set_title(title, fontsize=FONT_TITLE, fontweight='bold', pad=4*SCALE)

    # 5. Clean L-Shaped Spines (No Grid)
    ax.grid(False)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_linewidth(0.5 * SCALE)
    ax.spines['left'].set_linewidth(0.5 * SCALE)
    ax.tick_params(width=0.5 * SCALE, length=2 * SCALE, color="black")

# ── 6. Export ────────────────────────────────────────────────────────────────
pdf_path = "latency_boxplot_python.pdf"
svg_path = "latency_boxplot_python.svg"
png_path = "latency_boxplot_python.png"

plt.savefig(pdf_path, format='pdf', transparent=True, dpi=300)
plt.savefig(svg_path, format='svg', transparent=True)
plt.savefig(png_path, format='png', transparent=False, dpi=300, facecolor='white')

print(f"Saved: {pdf_path}, {svg_path}, and {png_path} at 3x scale. Ready for Illustrator.")

findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font f

Saved: latency_boxplot_python.pdf, latency_boxplot_python.svg, and latency_boxplot_python.png at 3x scale. Ready for Illustrator.
